**By:** Rafay Siddiqui  
**Student No:** 24K-0009  
**Section:** BAI-4A

In [ ]:
import pandas as pd
import numpy as np

# Set seed for reproducibility
np.random.seed(42)

def generate_student_data(n_samples):
    """Generates mock student performance data based on UCI dataset structure."""
    data = {
        'school': np.random.choice(['GP', 'MS'], n_samples, p=[0.8, 0.2]),
        'sex': np.random.choice(['F', 'M'], n_samples),
        'address': np.random.choice(['U', 'R'], n_samples, p=[0.75, 0.25]),
        'studytime': np.random.choice([1, 2, 3, 4], n_samples, p=[0.25, 0.45, 0.20, 0.10]),
        'failures': np.random.choice([0, 1, 2, 3], n_samples, p=[0.75, 0.15, 0.05, 0.05]),
        'G1': np.random.randint(5, 20, n_samples),
        'G2': np.random.randint(5, 20, n_samples)
    }
    df = pd.DataFrame(data)
    
    # G3 (final grade) is usually highly correlated with G1 and G2
    df['G3'] = (df['G1'] + df['G2']) / 2 + np.random.normal(0, 2, n_samples)
    df['G3'] = df['G3'].clip(0, 20).astype(int) 
    
    return df

# Generate datasets matching the shape described in the notebook
df_mat = generate_student_data(395)
df_por = generate_student_data(649)

# Save as CSV with ';' delimiter as expected by your EDA code
df_mat.to_csv('student-mat.csv', sep=';', index=False)
df_por.to_csv('student-por.csv', sep=';', index=False)

print("Files 'student-mat.csv' and 'student-por.csv' generated successfully!")

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mutual_info_score
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. IMPORT DATA & BASIC CHECKS
# ==========================================
print("--- Loading Data ---")
df_mat = pd.read_csv('student-mat.csv', delimiter=';')
df_por = pd.read_csv('student-por.csv', delimiter=';')

print(f"\nMath dataset shape: {df_mat.shape}")
print(f"Portuguese dataset shape: {df_por.shape}")

print("\n--- Missing Values ---")
print("Math NaNs:\n", df_mat.isna().sum())
print("Portuguese NaNs:\n", df_por.isna().sum())

print("\n--- Duplicates ---")
print("Math duplicates:", df_mat.duplicated().sum())
print("Portuguese duplicates:", df_por.duplicated().sum())

print("\n--- Data Types & Info (Math) ---")
df_mat.info()

print("\n--- Unique Values (Math) ---")
print(df_mat.nunique())

print("\n--- Statistical Description (Portuguese) ---")
print(df_por.describe())

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================
df_por['total score'] = df_por['G1'] + df_por['G2']
df_por['average'] = df_por['total score'] / 2

# ==========================================
# 3. MUTUAL INFORMATION SCORE
# ==========================================
print("\n--- Mutual Information Score (Categorical vs G3) ---")
def compute_mutual_information(categorical_serie):
    return mutual_info_score(categorical_serie, df_por.G3)

categorical_variables = df_por.select_dtypes(include=object)
feature_importance = categorical_variables.apply(compute_mutual_information).sort_values(ascending=False)
print(feature_importance)

# ==========================================
# 4. VISUALIZATIONS
# ==========================================
sns.set_theme(style="whitegrid")

# 4.1 Average Score Distribution
fig, axs = plt.subplots(1, 2, figsize=(15, 6))
sns.histplot(data=df_por, x='average', bins=30, kde=True, color='g', ax=axs[0])
axs[0].set_title('Average Score Distribution')
sns.histplot(data=df_por, x='average', kde=True, hue='sex', ax=axs[1])
axs[1].set_title('Average Score by Gender')
plt.show()

# 4.2 Study Time vs Average
plt.subplots(1, 3, figsize=(20, 6))
plt.subplot(131)
sns.histplot(data=df_por, x='average', kde=True, hue='studytime')
plt.title("Overall Study Time Impact")
plt.subplot(132)
sns.histplot(data=df_por[df_por.sex=='M'], x='average', kde=True, hue='studytime')
plt.title("Male Study Time Impact")
plt.subplot(133)
sns.histplot(data=df_por[df_por.sex=='F'], x='average', kde=True, hue='studytime')
plt.title("Female Study Time Impact")
plt.show()

# 4.3 Final Grade (G3) Distribution
plt.figure(figsize=(20, 6))
plt.subplot(131)
sns.histplot(data=df_por, x='G3', kde=True)
plt.title("Overall Final Grade (G3)")
plt.subplot(132)
sns.histplot(data=df_por[df_por['sex']=='F'], x='G3', kde=True)
plt.title("Female Final Grade (G3)")
plt.subplot(133)
sns.histplot(data=df_por[df_por['sex']=='M'], x='G3', kde=True)
plt.title("Male Final Grade (G3)")
plt.show()

# 4.4 Scores Violin Plot
plt.figure(figsize=(18, 6))
plt.subplot(1, 3, 1)
sns.violinplot(y='G1', data=df_por, linewidth=2, color='skyblue')
plt.title('G1 SCORES')
plt.subplot(1, 3, 2)
sns.violinplot(y='G2', data=df_por, linewidth=2, color='lightgreen')
plt.title('G2 SCORES')
plt.subplot(1, 3, 3)
sns.violinplot(y='G3', data=df_por, linewidth=2, color='salmon')
plt.title('G3 SCORES')
plt.show()

# 4.5 Multivariate Pie Charts
plt.figure(figsize=(20, 6))
plt.subplot(1, 3, 1)
size = df_por['sex'].value_counts()
plt.pie(size, labels=size.index, autopct='%1.2f%%')
plt.title('Gender (Sex)', fontsize=16)

plt.subplot(1, 3, 2)
size = df_por['school'].value_counts()
plt.pie(size, labels=size.index, autopct='%1.2f%%')
plt.title('School', fontsize=16)

plt.subplot(1, 3, 3)
size = df_por['address'].value_counts()
plt.pie(size, labels=size.index, autopct='%1.2f%%')
plt.title('Address (Urban/Rural)', fontsize=16)
plt.show()

# 4.6 Feature Wise (Gender)
f, ax = plt.subplots(1, 2, figsize=(15, 6))
sns.countplot(x='sex', data=df_por, palette='bright', ax=ax[0])
ax[0].set_title("Gender Count")
for container in ax[0].containers:
    ax[0].bar_label(container, color='black')

size = df_por['sex'].value_counts()
ax[1].pie(size, labels=size.index, explode=[0, 0.1], autopct='%1.1f%%', shadow=True)
ax[1].set_title("Gender Distribution")
plt.show()

# 4.7 Gender vs G1, G2, G3 Average Barplots
Group_data2 = df_por.groupby('sex')
f, ax = plt.subplots(1, 3, figsize=(20, 6))

sns.barplot(x=Group_data2['G1'].mean().index, y=Group_data2['G1'].mean().values, palette='mako', ax=ax[0])
ax[0].set_title('Avg G1 Score by Gender', size=16)

sns.barplot(x=Group_data2['G2'].mean().index, y=Group_data2['G2'].mean().values, palette='flare', ax=ax[1])
ax[1].set_title('Avg G2 Score by Gender', size=16)

sns.barplot(x=Group_data2['G3'].mean().index, y=Group_data2['G3'].mean().values, palette='coolwarm', ax=ax[2])
ax[2].set_title('Avg G3 Score by Gender', size=16)
plt.show()

# 4.8 School Distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='school', data=df_por, palette='Blues')
plt.title('Comparison of Schools', fontsize=16)
plt.show()

# 4.9 Study Time & Failures Breakdown
f, ax = plt.subplots(1, 2, figsize=(18, 6))
sns.countplot(x='studytime', data=df_por, hue='sex', palette='bright', ax=ax[0])
ax[0].set_title('Study Time vs Gender', size=16)

sns.countplot(x='failures', data=df_por, hue='sex', palette='bright', ax=ax[1])
ax[1].set_title('Failures vs Gender', size=16)
plt.show()

# 4.10 Scores vs Study Time by Gender
plt.figure(figsize=(18, 5))
plt.subplot(1, 3, 1)
sns.barplot(x='studytime', y='G1', hue='sex', data=df_por)
plt.title("G1 vs Studytime")

plt.subplot(1, 3, 2)
sns.barplot(x='studytime', y='G2', hue='sex', data=df_por)
plt.title("G2 vs Studytime")

plt.subplot(1, 3, 3)
sns.barplot(x='studytime', y='G3', hue='sex', data=df_por)
plt.title("G3 vs Studytime")

plt.tight_layout()
plt.show()